# Four Ways to Settle

In [Multilateral Settlement Networks](https://doi.org/10.26034/zh.ijccr.2025.8310) we explore the use of basic double entry bookkeeping (DEB) principles to deliver a universal base layer for settlement in Financial Networks of any size. This interactive Python notebook is designed to introduce the four ways to settle using DEB.

Let's start with an analysis of a payment with balance sheet representations: As Clavero explores in his article [Money and hierarchy: Four ways to discharge a payment obligation](https://dx.doi.org/10.2139/ssrn.4032398) there are other ways than simple 'transfer of title' or assignment of payer asset to settle an existing obligation.

We focus on the settlement of a single payment obligation of 100 from a Payer to a Payee. We'll assume that settling this obligation is always possible. All terminology is presented from the Payer's perspective.

Our goal is to gain a foundational understanding of these settlement methods using DEB. This understanding will be crucial as we progress towards analyzing network flow bookkeeping in more complex scenarios.

**Key Points:**

* We'll explore four ways a Payer can settle a payment obligation to a Payee.
* We'll analyze how these settlements impact both parties' accounts.
* We'll build a Python function to automate settlement calculations and show how basid bookkeping rules work.

By the end of this notebook, you'll have a solid grasp of the Four Ways to Settle concept and its bookkeeping implications. This knowledge will pave the way for understanding settlements within a larger network setting.

## Data framework

We will use Pandas data frames for Payer and Payee balance sheets, and combine them to observe the settlement flows.

In [1]:
import pandas as pd

# Initial balance sheets
payer_balance_sheet = pd.DataFrame({
    'Assets': ['A', ''],
    'Liabilities': ['B', -100]
})

payee_balance_sheet = pd.DataFrame({
    'Assets': ['C', -100],
    'Liabilities': ['D', '']
})

combined_df = pd.concat([payer_balance_sheet, payee_balance_sheet], axis=1)

# Create a list of tuples for the multi-level header
header_list = [('Payer', 'Assets'), ('Payer', 'Liabilities'), 
                ('Payee', 'Assets'), ('Payee', 'Liabilities')]
# Create the multi-level index
multi_index = pd.MultiIndex.from_tuples(header_list)
# Set the multi-index as columns for the combined DataFrame (assuming Method 1)
combined_df.columns = multi_index
# Assign row index directly using a list
combined_df.index = ['∆ funds', '∆ AC P/R']

print(combined_df)

          Payer              Payee            
         Assets Liabilities Assets Liabilities
∆ funds       A           B      C           D
∆ AC P/R               -100   -100            


Both Payer and Payee balance sheet have standard columns: Assets and Liabilities. For the purpose of studying the four ways to settle we only observe the balance sheet changes related to the discharge of Payer payment obligation.

The desired change is a reduced position in the Accounts Payable and Accounts Receivable for Payer and Payee. The corresponding line in the balance sheet is marked as '∆ AC P/R'.

The reduction of $-100$ (debit the liabilities of Payer and credit the assets of Payee) has to be balanced by entries in the line '∆ funds'. This is necessary to maintain the balance of the balance sheet. There are four positions in the two balance sheets that can be used to balance the discharge of the payment obligation marked with letters A, B, C, and D. Since every settlement involves two parties, and each party has two sides of a balance sheet to use for the settlement, we arrive at a two-by-two table with four ways to settle the obligation.

| Payment by | C +100     | D -100   |
|------------|------------|----------|
| **A -100** | assignment | set-off  |
| **B +100** | issuance   | novation |




## Four Ways to Settle Helper Function

This section dives into a Python function named four_ways that automates the settlement process based on available resources. We'll explore how the function tackles settling the 100 payment obligation using the four methods:

* **Set-off**: This prioritizes existing liabilities of the Payee towards the Payer.
* **Assignment**: This simulates a traditional payment where the Payer uses existing assets to settle the obligation (transfer of title).
* **Issuance**: Here, the Payer creates a new financial instrument (e.g., voucher) that becomes their liability and the Payee's asset.
* **Novation (Last Resort)**: This method comes into play when the previous methods fall short. Novation essentially involves the Payer taking over an existing liability that the Payee owes to a third party. In simpler terms, the Payer assumes the Payee's debt to another party to settle their own obligation.

**Understanding Novation**

It's important to acknowledge the complexity of novation, especially in traditional bilateral transaction settings. The involvement of a third party makes coordinating and verifying the transfer of obligations cumbersome. However, within a multilateral settlement network, novation becomes a more feasible option.

By understanding how novation works in this context, we gain valuable insight into how settlements can be achieved within a network, where interconnectedness can facilitate transactions that might be impractical in a one-on-one scenario.

The four_ways function prioritizes settlements using these methods in an ascending order of risk, with novation being the last resort due to its inherent complexities.


In [2]:
def four_ways(payee_existing_liability, payer_liquid_assets, payer_issuing_assets):

    # Initialise the balance sheet entries and the remaining amount of AC P/R to settle
    A = B = C = D = 0
    remaining = 100

    # Set off first
    set_off_amount = min(payee_existing_liability, remaining)
    A -= set_off_amount # Decrease Payer assets
    D -= set_off_amount # Decrease Payee liabilities
    remaining -= set_off_amount

    # Pay with assignment
    payment_amount = min(payer_liquid_assets, remaining)
    A -= payment_amount # Decrease Payer assets
    C += payment_amount # Increase Payee assets
    remaining -= payment_amount

    # Use issuance
    issued_amount = min(payer_issuing_assets, remaining)
    B += issued_amount # Increase Payer liabilities
    C += issued_amount # Increase Payee assets
    remaining -= issued_amount

    # Novate remaining
    novated_amount = remaining
    B += novated_amount # Increase Payer liabilities
    D -= novated_amount # Decrease Payee liabilities
    
    # Set up payer and payee balance sheets
    payer_balance_sheet = pd.DataFrame({
        'Assets': [A, ''],
        'Liabilities': [B, -100]
    })

    payee_balance_sheet = pd.DataFrame({
        'Assets': [C, -100],
        'Liabilities': [D, '']
    })

    combined_df = pd.concat([payer_balance_sheet, payee_balance_sheet], axis=1)

    # Create a list of tuples for the multi-level header
    header_list = [('Payer', 'Assets'), ('Payer', 'Liabilities'), 
                   ('Payee', 'Assets'), ('Payee', 'Liabilities')]
    # Create the multi-level index
    multi_index = pd.MultiIndex.from_tuples(header_list)
    # Set the multi-index as columns for the combined DataFrame (assuming Method 1)
    combined_df.columns = multi_index
    # Assign row index directly using a list
    combined_df.index = ['∆ funds', '∆ AC P/R']

    print()
    print(combined_df)

    print()
    print(f'Set_off:      {set_off_amount}')
    print(f'Assignment:   {payment_amount}')
    print(f'Issuance:     {issued_amount}')
    print(f'Novation:     {novated_amount}')
    print(f'-----------------')
    print(f'Settled:      {set_off_amount+payment_amount+issued_amount+novated_amount}')

    # Check for balanced transactions. ∆ of payer = ∆ of payee for all rows
    for i in range(len(combined_df)):
        row = combined_df.iloc[i]
        # Handle '' as 0 and use plus/minus to follow debtor creditor accounting rules
        payer_delta = -row[('Payer', 'Assets')] if not row[('Payer', 'Assets')] == '' else 0
        payee_delta = +row[('Payee', 'Assets')] if not row[('Payee', 'Assets')] == '' else 0
        payer_delta += row[('Payer', 'Liabilities')] if not row[('Payer', 'Liabilities')] == '' else 0
        payee_delta -= row[('Payee', 'Liabilities')] if not row[('Payee', 'Liabilities')] == '' else 0
        assert payer_delta == payee_delta, f'Unbalanced transaction in row {combined_df.index[i]}!'

    # Check for balanced payer and payee balance sheets
    payer_assets = combined_df[('Payer', 'Assets')]
    payer_liabilities = combined_df[('Payer', 'Liabilities')]
    payee_assets = combined_df[('Payee', 'Assets')]
    payee_liabilities = combined_df[('Payee', 'Liabilities')]
    # Handle '' as 0
    assert sum(x for x in payer_assets if not x == '') == sum(x for x in payer_liabilities if not x == ''), f'Payer balance sheet not balanced'
    assert sum(x for x in payee_assets if not x == '') == sum(x for x in payee_liabilities if not x == ''), f'Payee balance sheet not balanced'

With the "four_ways" function defined we can explore the four ways to settle individually and then in a combined manner.

## Set-off

This is a simple bilateral set-off example. An obligation to pay by Payer is set off by existing liability of Payee towards Payer.

In [3]:
payee_existing_liability = 100
payer_liquid_assets = 0
payer_issuing_assets = 0

four_ways(payee_existing_liability, payer_liquid_assets, payer_issuing_assets)


          Payer              Payee            
         Assets Liabilities Assets Liabilities
∆ funds    -100           0      0        -100
∆ AC P/R               -100   -100            

Set_off:      100
Assignment:   0
Issuance:     0
Novation:     0
-----------------
Settled:      100


## Assignment

This is a typical payment with a transfer of title. Assets used by Payer are a property of Payee after the transaction. 

In [4]:
payee_existing_liability = 0
payer_liquid_assets = 100
payer_issuing_assets = 0

four_ways(payee_existing_liability, payer_liquid_assets, payer_issuing_assets)


          Payer              Payee            
         Assets Liabilities Assets Liabilities
∆ funds    -100           0    100           0
∆ AC P/R               -100   -100            

Set_off:      0
Assignment:   100
Issuance:     0
Novation:     0
-----------------
Settled:      100


## Issuance

In this case Payer issues a new financial instrument that is Payer liability and Payee asset. Voucher for example!

In [5]:
payee_existing_liability = 0
payer_liquid_assets = 0
payer_issuing_assets = 100

four_ways(payee_existing_liability, payer_liquid_assets, payer_issuing_assets)


          Payer              Payee            
         Assets Liabilities Assets Liabilities
∆ funds       0         100    100           0
∆ AC P/R               -100   -100            

Set_off:      0
Assignment:   0
Issuance:     100
Novation:     0
-----------------
Settled:      100


## Novation

In this case Payer settles by taking over the existing Payee liability.

In [6]:
payee_existing_liability = 0
payer_liquid_assets = 0
payer_issuing_assets = 0

four_ways(payee_existing_liability, payer_liquid_assets, payer_issuing_assets)


          Payer              Payee            
         Assets Liabilities Assets Liabilities
∆ funds       0         100      0        -100
∆ AC P/R               -100   -100            

Set_off:      0
Assignment:   0
Issuance:     0
Novation:     100
-----------------
Settled:      100


## Free style - Try for yourself

In real life and especially in a large network setting, there will be a mix of various ways to settle.

**Try changing the available resources!**

In [7]:
payee_existing_liability = 10     # Does Payee owe something to Payer? Any positive integer is OK!
payer_liquid_assets = 20          # Available liquid assets of Payer
payer_issuing_assets = 10         # Payer credit line, new vouchers, new mutual credit

four_ways(payee_existing_liability, payer_liquid_assets, payer_issuing_assets)


          Payer              Payee            
         Assets Liabilities Assets Liabilities
∆ funds     -30          70     30         -70
∆ AC P/R               -100   -100            

Set_off:      10
Assignment:   20
Issuance:     10
Novation:     60
-----------------
Settled:      100
